In [1]:
import nltk
import numpy as np
import pandas as pd

from datasets import load_dataset
from nltk.tokenize import TweetTokenizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt

In [2]:
dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")

print(dataset)
print(dataset["train"][:5])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})
{'text': ['"QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"', '"Ben Smith / Smith (concussion) remains out of the lineup Thursday, Curtis #NHL #SJ"', 'Sorry bout the stream last night I crashed out but will be on tonight for sure. Then back to Minecraft in pc tomorrow night.', "Chase Headley's RBI double in the 8th inning off David Price snapped a Yankees streak of 33 consecutive scoreless innings against Blue Jays", '@user Alciato: Bee will invest 150 million in January, another 200 in the Summer and plans to bring Messi by 2017"'], 'label': [2, 1, 1, 1, 2]}


In [3]:
train_df = pd.DataFrame(dataset["train"])
validation_df = pd.DataFrame(dataset["validation"])
test_df = pd.DataFrame(dataset["test"])

print(train_df.head())

                                                text  label
0  "QT @user In the original draft of the 7th boo...      2
1  "Ben Smith / Smith (concussion) remains out of...      1
2  Sorry bout the stream last night I crashed out...      1
3  Chase Headley's RBI double in the 8th inning o...      1
4  @user Alciato: Bee will invest 150 million in ...      2


In [4]:
class_names = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

class_counts = train_df["label"].value_counts().sort_index()
# print(class_counts)
for label, count in class_counts.items():
    print(f"{class_names[label]}: {count}")

Negative: 7093
Neutral: 20673
Positive: 17849


In [5]:
class_frequency = {
    class_names[label]: count
    for label, count in class_counts.items()
}

print(class_frequency)

{'Negative': 7093, 'Neutral': 20673, 'Positive': 17849}


In [6]:
tokenizer = TweetTokenizer(
    preserve_case=False,
    strip_handles=True,
    reduce_len=True
)

def preprocess_tweet(tweet):
    tokens = tokenizer.tokenize(tweet)

    tokens = [
        token
        for token in tokens
        if token.isalpha()
    ]

    return " ".join(tokens)


train_df["processed_text"] = train_df["text"].apply(preprocess_tweet)
test_df["processed_text"] = test_df["text"].apply(preprocess_tweet)

In [7]:
X_train = train_df["processed_text"]
y_train = train_df["label"]

X_test = test_df["processed_text"]
y_test = test_df["label"]

In [8]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training shape:", X_train_tfidf.shape)
print("Testing shape:", X_test_tfidf.shape)

Training shape: (45615, 10000)
Testing shape: (12284, 10000)


In [9]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_tfidf, y_train)

print("Training complete!")

Training complete!


In [10]:
y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.5859


In [11]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ]
    )
)

              precision    recall  f1-score   support

    Negative       0.67      0.35      0.46      3972
     Neutral       0.59      0.74      0.66      5937
    Positive       0.50      0.60      0.55      2375

    accuracy                           0.59     12284
   macro avg       0.59      0.56      0.56     12284
weighted avg       0.60      0.59      0.57     12284



In [12]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[1402 2130  440]
 [ 601 4369  967]
 [  88  861 1426]]


In [13]:
def predict_sentiment(tweet):

    processed = preprocess_tweet(tweet)

    features = vectorizer.transform([processed])

    prediction = model.predict(features)[0]

    probabilities = model.predict_proba(features)[0]

    sentiment = class_names[prediction]

    confidence = probabilities[prediction]

    return sentiment, confidence, probabilities

test_tweets = [
    "I absolutely love this!",
    "This is terrible.",
    "The movie was okay, nothing special.",
    "I don't really know how I feel about this.",
    "This is the best thing ever!"
]

for tweet in test_tweets:

    sentiment, confidence, probabilities = predict_sentiment(tweet)

    print(f"Tweet: {tweet}")
    print(f"Sentiment: {sentiment}")
    print(f"Confidence: {confidence:.2%}")
    print()

Tweet: I absolutely love this!
Sentiment: Positive
Confidence: 97.01%

Tweet: This is terrible.
Sentiment: Negative
Confidence: 92.46%

Tweet: The movie was okay, nothing special.
Sentiment: Neutral
Confidence: 51.44%

Tweet: I don't really know how I feel about this.
Sentiment: Neutral
Confidence: 53.39%

Tweet: This is the best thing ever!
Sentiment: Positive
Confidence: 96.69%



In [14]:
from nltk.stem import PorterStemmer
import re
import string
from nltk.corpus import stopwords


def process_tweet(tweet):
    """Process tweet function.
    Input:
        tweet: a string containing a tweet
    Output:
        tweets_clean: a list of words containing the processed tweet

    """
    stemmer = PorterStemmer()
    stopwords_english = stopwords.words('english')
    # remove stock market tickers like $GE
    tweet = re.sub(r'\$\w*', '', tweet)
    # remove old style retweet text "RT"
    tweet = re.sub(r'^RT[\s]+', '', tweet)
    # remove hyperlinks    
    tweet = re.sub(r'https?://[^\s\n\r]+', '', tweet)
    # remove hashtags
    # only removing the hash # sign from the word
    tweet = re.sub(r'#', '', tweet)
    # tokenize tweets
    tokenizer = TweetTokenizer(preserve_case=False, strip_handles=True,
                               reduce_len=True)
    tweet_tokens = tokenizer.tokenize(tweet)

    tweets_clean = []
    for word in tweet_tokens:
        if (word not in stopwords_english and  # remove stopwords
                word not in string.punctuation):  # remove punctuation
            # tweets_clean.append(word)
            stem_word = stemmer.stem(word)  # stemming word
            tweets_clean.append(stem_word)

    return tweets_clean

In [15]:
while True:

	tweet = input("\nEnter a tweet (or type 'quit' to exit): ")
	
	if tweet.lower() == "quit":
		print("Goodbye!")
		break
	
	
	sentiment, confidence, probabilities = predict_sentiment(tweet)
	
	print(f"\nUser input: {tweet}")
	print("Prediction")
	print("-" * 30)
	print(preprocess_tweet(tweet=tweet))
	print(f"Sentiment: {sentiment}")
	print(f"Confidence: {confidence:.2%}")

	print("\nClass probabilities:")

	for label, probability in zip(
		["Negative", "Neutral", "Positive"],
		probabilities
	):
		print(f"{label}: {probability:.2%}")


User input: I don't love it at all!!!!
Prediction
------------------------------
i love it at all
Sentiment: Positive
Confidence: 94.41%

Class probabilities:
Negative: 2.02%
Neutral: 3.57%
Positive: 94.41%

User input: The movie was not that greate
Prediction
------------------------------
the movie was not that greate
Sentiment: Neutral
Confidence: 67.23%

Class probabilities:
Negative: 26.51%
Neutral: 67.23%
Positive: 6.26%

User input: The music was really beautiful
Prediction
------------------------------
the music was really beautiful
Sentiment: Positive
Confidence: 80.05%

Class probabilities:
Negative: 7.05%
Neutral: 12.90%
Positive: 80.05%
Goodbye!
